# 🔍 Módulo 4 — Análise Exploratória de Dados (EDA)

**Curso Introdutório de Python para Ciência de Dados**  
Universidade de Fortaleza — UNIFOR | Disciplina T326

---

## O que você vai aprender neste módulo?

| Tópico | Descrição |
|--------|----------|
| 🎯 Definição do problema | Como formular perguntas de negócio úteis e respondíveis |
| 🗺️ Exploração inicial | Primeiros passos: entender a estrutura, qualidade e distribuição dos dados |
| 📊 Análise univariada | Explorar cada variável individualmente |
| 🔗 Análise bivariada e multivariada | Descobrir relações entre variáveis |
| 📖 Storytelling com dados | Como narrar insights de forma clara e convincente |
| ✅ Conclusões | Sintetizar descobertas em linguagem acessível |

---

## 🗄️ Dataset: Brazilian Cities Dataset

Usaremos dados socioeconômicos dos **5.570 municípios brasileiros**, incluindo:
- **IDHM** e seus componentes (Renda, Longevidade, Educação)
- PIB e PIB per capita
- População, área e densidade demográfica
- Região e estado

> **Fonte:** IBGE / Atlas Brasil — dados do último Censo disponível.

---

**❓ Pergunta central desta EDA:**
> *"O que determina o nível de desenvolvimento humano (IDHM) de um município brasileiro, e como ele varia entre regiões, estados e categorias urbanas?"*

---
## ⚙️ 0. Configuração do Ambiente

Antes de qualquer análise, importamos as bibliotecas que usaremos e configuramos o visual dos gráficos. Essa é uma boa prática: centralizar todas as importações no início do notebook.

In [ ]:
# --- Importações ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

# Suprimir avisos desnecessários
warnings.filterwarnings('ignore')

# --- Configuração de estilo ---
# Definimos um tema visual consistente para todos os gráficos do notebook
plt.rcParams.update({
    'figure.dpi': 120,
    'figure.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
})
sns.set_palette('Set2')

# Paleta de cores por região — usaremos ao longo de toda a análise
COR_REGIAO = {
    'Norte':     '#1f77b4',
    'Nordeste':  '#ff7f0e',
    'Centro-Oeste': '#2ca02c',
    'Sudeste':   '#d62728',
    'Sul':       '#9467bd'
}

print('✅ Ambiente configurado com sucesso!')

---
## 📥 1. Carregamento e Visão Geral dos Dados

### 1.1 Carregando o dataset

O dataset pode ser baixado do Kaggle: **Brazilian Cities Dataset**.  
Para este notebook, assumimos que o arquivo `brazil_cities.csv` está na pasta `../dataset/`.

> **Dica pedagógica:** Sempre verifique se o arquivo carregou corretamente antes de qualquer operação.

In [ ]:
# Carregando o dataset
# O encoding='latin-1' é necessário para caracteres especiais do português
try:
    df = pd.read_csv('../dataset/brazil_cities.csv', encoding='latin-1', sep=';')
    print(f'✅ Dataset carregado com sucesso!')
    print(f'   Linhas    : {df.shape[0]:,}')
    print(f'   Colunas   : {df.shape[1]}')
except FileNotFoundError:
    print('⚠️  Arquivo não encontrado. Gerando dataset sintético para demonstração...')
    # Geramos dados sintéticos realistas para fins didáticos
    np.random.seed(42)
    n = 5570
    regioes = ['Norte', 'Nordeste', 'Centro-Oeste', 'Sudeste', 'Sul']
    estados_por_regiao = {
        'Norte': ['AM', 'PA', 'AC', 'RO', 'RR', 'AP', 'TO'],
        'Nordeste': ['MA', 'PI', 'CE', 'RN', 'PB', 'PE', 'AL', 'SE', 'BA'],
        'Centro-Oeste': ['MT', 'MS', 'GO', 'DF'],
        'Sudeste': ['MG', 'ES', 'RJ', 'SP'],
        'Sul': ['PR', 'SC', 'RS']
    }
    # Médias e desvios-padrão de IDHM por região (baseados em dados reais)
    idhm_params = {
        'Norte': (0.620, 0.05),
        'Nordeste': (0.600, 0.055),
        'Centro-Oeste': (0.700, 0.045),
        'Sudeste': (0.735, 0.05),
        'Sul': (0.745, 0.04)
    }
    regiao_list, estado_list, idhm_list = [], [], []
    pib_pc_list, pop_list, area_list = [], [], []
    idhm_r_list, idhm_l_list, idhm_e_list = [], [], []
    
    for regiao, (mu, sigma) in idhm_params.items():
        estados = estados_por_regiao[regiao]
        n_r = int(n * [0.12, 0.35, 0.10, 0.22, 0.21][regioes.index(regiao)])
        for _ in range(n_r):
            idhm = np.clip(np.random.normal(mu, sigma), 0.4, 0.95)
            regiao_list.append(regiao)
            estado_list.append(np.random.choice(estados))
            idhm_list.append(round(idhm, 3))
            # Componentes do IDHM variam em torno do IDHM total
            idhm_r_list.append(round(np.clip(idhm + np.random.normal(0, 0.03), 0.35, 0.95), 3))
            idhm_l_list.append(round(np.clip(idhm + np.random.normal(0.02, 0.02), 0.4, 0.95), 3))
            idhm_e_list.append(round(np.clip(idhm - np.random.normal(0.02, 0.03), 0.3, 0.95), 3))
            # PIB per capita correlacionado com IDHM
            pib_pc = max(3000, idhm * 60000 + np.random.normal(0, 8000))
            pib_pc_list.append(round(pib_pc, 2))
            # População: maioria de municípios pequenos
            pop_list.append(int(np.random.lognormal(9, 1.5)))
            area_list.append(round(np.random.lognormal(7, 1.2), 1))
    
    # Ajuste para ter exatamente n linhas
    total = len(regiao_list)
    df = pd.DataFrame({
        'CITY': [f'Municipio_{i:04d}' for i in range(total)],
        'STATE': estado_list,
        'REGION': regiao_list,
        'IDHM': idhm_list,
        'IDHM_Renda': idhm_r_list,
        'IDHM_Longevidade': idhm_l_list,
        'IDHM_Educacao': idhm_e_list,
        'GDP_CAPITA': pib_pc_list,
        'IBGE_POP': pop_list,
        'AREA': area_list,
    })
    df['DENSIDADE'] = (df['IBGE_POP'] / df['AREA']).round(2)
    df['GDP'] = (df['GDP_CAPITA'] * df['IBGE_POP']).round(0).astype(int)
    print(f'✅ Dataset sintético criado: {df.shape[0]} municípios, {df.shape[1]} colunas')

### 1.2 Primeiras impressões

Antes de qualquer análise, precisamos conhecer o terreno: quais colunas existem, que tipos de dados são, e se há valores ausentes.

In [ ]:
# Exibir as primeiras 5 linhas
# .head() nos dá um "cartão de visitas" do dataset
print('=== PRIMEIRAS 5 LINHAS ===')
df.head()

In [ ]:
# Resumo estrutural do dataset
print('=== TIPOS DE DADOS E VALORES NÃO NULOS ===')
df.info()

In [ ]:
# Estatísticas descritivas das variáveis numéricas
# count, mean, std, min, 25%, 50%, 75%, max
print('=== ESTATÍSTICAS DESCRITIVAS ===')
df.describe().round(3)

In [ ]:
# Verificação de valores ausentes
# Uma EDA responsável sempre começa entendendo a qualidade dos dados!
nulos = df.isnull().sum()
pct_nulos = (df.isnull().mean() * 100).round(2)

resumo_nulos = pd.DataFrame({'Valores Nulos': nulos, '% do Total': pct_nulos})
resumo_nulos = resumo_nulos[resumo_nulos['Valores Nulos'] > 0].sort_values('% do Total', ascending=False)

if resumo_nulos.empty:
    print('✅ Sem valores ausentes! Dataset limpo.')
else:
    print('⚠️  Colunas com valores ausentes:')
    print(resumo_nulos)

---
## 📊 2. Análise Univariada — Cada Variável Por Si

> **Conceito-chave:** A análise univariada estuda *uma variável de cada vez*. O objetivo é entender sua distribuição, tendência central, dispersão e presença de outliers.

### 2.1 Distribuição do IDHM — a variável-alvo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Histograma ---
ax = axes[0]
ax.hist(df['IDHM'].dropna(), bins=40, color='steelblue', edgecolor='white', alpha=0.85)
media = df['IDHM'].mean()
mediana = df['IDHM'].median()
ax.axvline(media, color='#d62728', linestyle='--', linewidth=2, label=f'Média: {media:.3f}')
ax.axvline(mediana, color='#ff7f0e', linestyle='-', linewidth=2, label=f'Mediana: {mediana:.3f}')
ax.set_title('Distribuição do IDHM nos Municípios Brasileiros', fontsize=13, fontweight='bold')
ax.set_xlabel('IDHM')
ax.set_ylabel('Número de Municípios')
ax.legend()

# Anotação dos intervalos do IDHM (classificação ONU)
for inicio, fim, label, cor in [(0.0, 0.499, 'Muito Baixo', '#fee5d9'),
                                  (0.5, 0.599, 'Baixo', '#fcae91'),
                                  (0.6, 0.699, 'Médio', '#fb6a4a'),
                                  (0.7, 0.799, 'Alto', '#de2d26'),
                                  (0.8, 1.0,   'Muito Alto', '#a50f15')]:
    ax.axvspan(inicio, fim, alpha=0.07, color=cor)

# --- Boxplot ---
ax2 = axes[1]
ax2.boxplot(df['IDHM'].dropna(), vert=True, patch_artist=True,
            boxprops=dict(facecolor='steelblue', alpha=0.6),
            medianprops=dict(color='#d62728', linewidth=2))
ax2.set_title('Boxplot do IDHM\n(outliers = municípios extremos)', fontsize=13, fontweight='bold')
ax2.set_ylabel('IDHM')
ax2.set_xticks([])

# Adicionar quartis como anotações
q1 = df['IDHM'].quantile(0.25)
q3 = df['IDHM'].quantile(0.75)
ax2.annotate(f'Q1: {q1:.3f}', xy=(1, q1), xytext=(1.3, q1), fontsize=9, va='center')
ax2.annotate(f'Q3: {q3:.3f}', xy=(1, q3), xytext=(1.3, q3), fontsize=9, va='center')

plt.suptitle('📊 Análise Univariada: IDHM', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../modulo-4/figuras/fig1_distribuicao_idhm.png', bbox_inches='tight')
plt.show()

# --- Interpretação ---
print('\n📖 INTERPRETAÇÃO:')
print(f'  Média do IDHM nacional: {media:.3f}')
print(f'  Mediana              : {mediana:.3f}')
print(f'  Desvio-padrão        : {df["IDHM"].std():.3f}')
print(f'  Menor IDHM           : {df["IDHM"].min():.3f}')
print(f'  Maior IDHM           : {df["IDHM"].max():.3f}')
print()
print('  A distribuição é ligeiramente assimétrica à esquerda:')
print('  a maioria dos municípios se concentra entre 0,55 e 0,75 (IDHM Baixo-Médio).')
print('  A média > mediana indica que municípios com IDHM muito alto puxam a média para cima.')

### 2.2 Classificando municípios por faixa de IDHM

A ONU classifica o IDHM em faixas. Vamos ver como os municípios brasileiros se distribuem.

In [ ]:
# Criando a coluna de faixa de IDHM
bins   = [0, 0.499, 0.599, 0.699, 0.799, 1.0]
labels = ['Muito Baixo\n(< 0,500)', 'Baixo\n(0,500–0,599)',
          'Médio\n(0,600–0,699)', 'Alto\n(0,700–0,799)', 'Muito Alto\n(≥ 0,800)']
df['FAIXA_IDHM'] = pd.cut(df['IDHM'], bins=bins, labels=labels, include_lowest=True)

# Contagem e percentual por faixa
contagem = df['FAIXA_IDHM'].value_counts().sort_index()
percentual = (contagem / len(df) * 100).round(1)

# Gráfico de barras horizontais
cores_faixa = ['#de77ae', '#f1b6da', '#fee090', '#a6d96a', '#1a9641']
fig, ax = plt.subplots(figsize=(10, 5))
barras = ax.barh(contagem.index, contagem.values, color=cores_faixa, edgecolor='white')

# Adicionar rótulos nas barras
for barra, pct in zip(barras, percentual.values):
    ax.text(barra.get_width() + 20, barra.get_y() + barra.get_height()/2,
            f'{barra.get_width():,.0f} municípios ({pct}%)',
            va='center', fontsize=10)

ax.set_title('Distribuição dos Municípios por Faixa de IDHM', fontsize=13, fontweight='bold')
ax.set_xlabel('Número de Municípios')
ax.set_xlim(0, contagem.max() * 1.35)
plt.tight_layout()
plt.savefig('../modulo-4/figuras/fig2_faixas_idhm.png', bbox_inches='tight')
plt.show()

print('\n📖 INSIGHT: A maioria dos municípios brasileiros se encontra nas faixas')
print('   Baixo e Médio de IDHM, evidenciando um desafio estrutural de desenvolvimento.')

---
## 🗺️ 3. Análise Geográfica — Onde Estão as Disparidades?

> **Pergunta norteadora nº 1:** *Como o IDHM varia entre as cinco regiões do Brasil? Quais estados têm os maiores e menores índices?*

In [ ]:
# Estatísticas do IDHM por Região
# .agg() permite calcular múltiplas estatísticas de uma vez
idhm_regiao = (
    df.groupby('REGION')['IDHM']
    .agg(Media='mean', Mediana='median', Desvio='std', Minimo='min', Maximo='max', N_Municipios='count')
    .round(3)
    .sort_values('Media', ascending=False)
)
print('=== IDHM POR REGIÃO ===')
print(idhm_regiao.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# --- Gráfico 1: Boxplot por Região ---
ax = axes[0]
ordem = df.groupby('REGION')['IDHM'].median().sort_values(ascending=False).index.tolist()

dados_boxplot = [df[df['REGION'] == r]['IDHM'].dropna().values for r in ordem]
bp = ax.boxplot(dados_boxplot, labels=ordem, patch_artist=True,
                medianprops=dict(color='black', linewidth=2))

for patch, regiao in zip(bp['boxes'], ordem):
    patch.set_facecolor(COR_REGIAO[regiao])
    patch.set_alpha(0.7)

ax.set_title('Distribuição do IDHM por Região', fontsize=12, fontweight='bold')
ax.set_ylabel('IDHM')
ax.set_xlabel('Região')
ax.tick_params(axis='x', rotation=15)

# --- Gráfico 2: IDHM médio por Estado (top 10 e bottom 10) ---
ax2 = axes[1]
idhm_estado = df.groupby('STATE')['IDHM'].mean().sort_values()

# Top 5 e Bottom 5
top5    = idhm_estado.tail(5)
bottom5 = idhm_estado.head(5)
destaque = pd.concat([bottom5, top5])

cores_bar = ['#d62728']*5 + ['#2ca02c']*5
barras = ax2.barh(destaque.index, destaque.values, color=cores_bar, edgecolor='white')
ax2.axvline(df['IDHM'].mean(), color='gray', linestyle='--', linewidth=1.5, label='Média nacional')

for barra in barras:
    ax2.text(barra.get_width() + 0.003, barra.get_y() + barra.get_height()/2,
             f'{barra.get_width():.3f}', va='center', fontsize=9)

ax2.set_title('Top 5 e Bottom 5 Estados\npor IDHM Médio', fontsize=12, fontweight='bold')
ax2.set_xlabel('IDHM Médio')
ax2.legend()

plt.suptitle('🗺️ Pergunta 1: Distribuição Geográfica do IDHM', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../modulo-4/figuras/fig3_idhm_por_regiao_estado.png', bbox_inches='tight')
plt.show()

print('\n📖 NARRATIVA:')
print('   Sul e Sudeste lideram o IDHM médio, enquanto Norte e Nordeste apresentam')
print('   os menores índices e maior variação interna (caixas mais largas no boxplot).')
print('   Isso revela desigualdade tanto ENTRE regiões quanto DENTRO delas.')

---
## 🔗 4. Análise Bivariada — Relações Entre Variáveis

> **Pergunta norteadora nº 2:** *Quais variáveis (PIB per capita, densidade) têm maior correlação com o IDHM?*

In [ ]:
# Matriz de correlação — visão geral das relações lineares entre variáveis
# Usamos apenas colunas numéricas relevantes
cols_numericas = ['IDHM', 'IDHM_Renda', 'IDHM_Longevidade', 'IDHM_Educacao', 
                  'GDP_CAPITA', 'IBGE_POP', 'DENSIDADE']

# Filtra apenas colunas que existem no dataframe
cols_numericas = [c for c in cols_numericas if c in df.columns]
corr = df[cols_numericas].corr().round(3)

fig, ax = plt.subplots(figsize=(9, 7))
mascara = np.triu(np.ones_like(corr, dtype=bool), k=1)  # Oculta triângulo superior (redundante)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            mask=mascara, ax=ax, linewidths=0.5,
            vmin=-1, vmax=1, square=True,
            cbar_kws={'label': 'Correlação de Pearson'})
ax.set_title('Matriz de Correlação entre Variáveis Socioeconômicas', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../modulo-4/figuras/fig4_correlacao.png', bbox_inches='tight')
plt.show()

print('\n📖 COMO LER O HEATMAP:')
print('   🟢 Verde escuro (próximo de +1) → forte correlação positiva')
print('   🔴 Vermelho escuro (próximo de -1) → forte correlação negativa')
print('   ⬜ Próximo de 0 → sem correlação linear')
print()
print('   Destaque: GDP_CAPITA apresenta correlação FORTE e POSITIVA com IDHM.')
print('   Quanto maior o PIB per capita, maior tende a ser o IDHM do município.')

In [ ]:
# Scatter plot: PIB per capita x IDHM (colorido por região)
# Esse é um dos gráficos mais poderosos para comunicar relação + contexto

fig, ax = plt.subplots(figsize=(12, 7))

for regiao, grupo in df.groupby('REGION'):
    ax.scatter(
        grupo['GDP_CAPITA'], grupo['IDHM'],
        label=regiao, color=COR_REGIAO[regiao],
        alpha=0.45, s=18, edgecolors='none'
    )

# Linha de tendência geral (regressão linear)
from numpy.polynomial.polynomial import polyfit
dados_validos = df[['GDP_CAPITA', 'IDHM']].dropna()
x_plot = np.linspace(dados_validos['GDP_CAPITA'].min(), dados_validos['GDP_CAPITA'].quantile(0.99), 200)
coefs = np.polyfit(dados_validos['GDP_CAPITA'], dados_validos['IDHM'], 1)
ax.plot(x_plot, np.polyval(coefs, x_plot), 'k--', linewidth=2, label='Tendência geral', zorder=5)

ax.set_xlabel('PIB per Capita (R$)', fontsize=11)
ax.set_ylabel('IDHM', fontsize=11)
ax.set_title('PIB per Capita × IDHM por Município e Região', fontsize=13, fontweight='bold')
ax.legend(title='Região', framealpha=0.8)
ax.set_xlim(left=0)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R$ {x/1000:.0f}k'))

plt.tight_layout()
plt.savefig('../modulo-4/figuras/fig5_scatter_pib_idhm.png', bbox_inches='tight')
plt.show()

print('\n📖 NARRATIVA:')
print('   A relação entre PIB per capita e IDHM é positiva, mas não perfeitamente linear.')
print('   Municípios do Sul/Sudeste (vermelho/roxo) dominam as posições de maior PIB e IDHM.')
print('   Municípios do Nordeste (laranja) aparecem concentrados no canto inferior esquerdo.')
print('   Existe uma dispersão considerável: há municípios com PIB elevado mas IDHM baixo')
print('   (riqueza concentrada que não se traduz em bem-estar coletivo).')

---
## ⚖️ 5. Desigualdade Interna — A "Cauda Longa" do Desenvolvimento

> **Pergunta norteadora nº 3:** *Qual o grau de desigualdade no IDHM entre municípios do mesmo estado?*

In [ ]:
# Desvio-padrão do IDHM por estado = medida de desigualdade interna
# Um estado com desvio alto tem municípios MUITO diferentes entre si
desigualdade = (
    df.groupby(['STATE', 'REGION'])['IDHM']
    .agg(Media='mean', DesvPad='std', Min='min', Max='max')
    .reset_index()
    .sort_values('DesvPad', ascending=False)
).round(3)

fig, ax = plt.subplots(figsize=(13, 6))

# Colorir barras pela região
cores = [COR_REGIAO[r] for r in desigualdade['REGION']]
barras = ax.bar(desigualdade['STATE'], desigualdade['DesvPad'], color=cores, edgecolor='white', alpha=0.85)

ax.set_title('Desigualdade Interna do IDHM por Estado\n(Desvio-Padrão entre municípios do mesmo estado)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Estado (UF)')
ax.set_ylabel('Desvio-Padrão do IDHM')
ax.tick_params(axis='x', rotation=60)

# Legenda de regiões
from matplotlib.patches import Patch
handles = [Patch(facecolor=cor, label=reg) for reg, cor in COR_REGIAO.items()]
ax.legend(handles=handles, title='Região', loc='upper right')

plt.tight_layout()
plt.savefig('../modulo-4/figuras/fig6_desigualdade_estados.png', bbox_inches='tight')
plt.show()

top3 = desigualdade.head(3)
print('\n📖 ESTADOS COM MAIOR DESIGUALDADE INTERNA:')
for _, row in top3.iterrows():
    print(f'   {row["STATE"]} ({row["REGION"]}): desvio-padrão = {row["DesvPad"]:.3f}, '
          f'range = {row["Min"]:.3f} – {row["Max"]:.3f}')
print('\n   Isso significa que dentro de um mesmo estado existem municípios com')
print('   níveis de desenvolvimento completamente distintos — ricos e pobres lado a lado.')

---
## 🏙️ 6. Tamanho Populacional × Desenvolvimento

> **Pergunta norteadora nº 6:** *Municípios maiores são necessariamente mais desenvolvidos?*

In [ ]:
# Categorizando municípios por porte populacional
# Classificação inspirada no critério do IBGE
bins_pop  = [0, 5000, 20000, 100000, 500000, float('inf')]
labels_pop = ['Muito pequeno\n(<5k)', 'Pequeno\n(5k–20k)', 
               'Médio\n(20k–100k)', 'Grande\n(100k–500k)', 'Metrópole\n(>500k)']
df['PORTE'] = pd.cut(df['IBGE_POP'], bins=bins_pop, labels=labels_pop)

# IDHM médio por porte
idhm_porte = df.groupby('PORTE', observed=True)['IDHM'].agg(['mean', 'count', 'std']).reset_index()
idhm_porte.columns = ['PORTE', 'IDHM_Medio', 'N_Municipios', 'Desvio']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: IDHM médio por porte
ax = axes[0]
cores_porte = sns.color_palette('Blues', n_colors=len(idhm_porte))
barras = ax.bar(idhm_porte['PORTE'], idhm_porte['IDHM_Medio'],
                color=cores_porte, edgecolor='white',
                yerr=idhm_porte['Desvio'], capsize=4)
ax.set_title('IDHM Médio por Porte do Município', fontsize=12, fontweight='bold')
ax.set_ylabel('IDHM Médio')
ax.set_ylim(0.5, 0.85)
for barra, val in zip(barras, idhm_porte['IDHM_Medio']):
    ax.text(barra.get_x() + barra.get_width()/2, barra.get_height() + 0.01,
            f'{val:.3f}', ha='center', fontsize=9)

# Gráfico 2: Número de municípios por porte
ax2 = axes[1]
ax2.bar(idhm_porte['PORTE'], idhm_porte['N_Municipios'], color=cores_porte, edgecolor='white')
ax2.set_title('Quantidade de Municípios por Porte', fontsize=12, fontweight='bold')
ax2.set_ylabel('Número de Municípios')
for barra, val in zip(ax2.patches, idhm_porte['N_Municipios']):
    ax2.text(barra.get_x() + barra.get_width()/2, barra.get_height() + 20,
             f'{val:,}', ha='center', fontsize=9)

plt.suptitle('🏙️ Pergunta 6: Tamanho Populacional e Desenvolvimento', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../modulo-4/figuras/fig7_porte_idhm.png', bbox_inches='tight')
plt.show()

print('\n📖 NARRATIVA:')
print('   A grande maioria dos municípios brasileiros é de porte pequeno (<20k hab.).')
print('   Municípios maiores tendem a ter IDHM mais alto, mas existem exceções notáveis:')
print('   pequenos municípios do interior do Sul e Sudeste alcançam IDHM elevado,')
print('   enquanto grandes centros do Norte/Nordeste ficam abaixo da média.')

---
## 🧩 7. Análise dos Componentes do IDHM — Onde Está o Gargalo?

> **Pergunta norteadora nº 5:** *Qual dos três componentes do IDHM (Renda, Longevidade, Educação) é o "gargalo" em cada região?*

O IDHM é composto por três dimensões:
- 📚 **Educação**: Escolaridade e frequência escolar
- 💰 **Renda**: PIB per capita ajustado
- ❤️ **Longevidade**: Expectativa de vida ao nascer

In [ ]:
# Calculamos a média de cada componente por região
cols_componentes = ['IDHM_Renda', 'IDHM_Longevidade', 'IDHM_Educacao']
cols_componentes = [c for c in cols_componentes if c in df.columns]

componentes_regiao = (
    df.groupby('REGION')[cols_componentes]
    .mean()
    .round(3)
    .loc[['Norte', 'Nordeste', 'Centro-Oeste', 'Sudeste', 'Sul']]
)

# Renomear colunas para legenda mais limpa
componentes_regiao.columns = ['Renda', 'Longevidade', 'Educação']

# Gráfico de barras agrupadas
fig, ax = plt.subplots(figsize=(13, 6))
componentes_regiao.plot(kind='bar', ax=ax, width=0.75,
                         color=['#e6994c', '#56b4e9', '#009e73'],
                         edgecolor='white')

ax.set_title('Componentes do IDHM por Região\n(Renda, Longevidade, Educação)', 
             fontsize=13, fontweight='bold')
ax.set_xlabel('Região')
ax.set_ylabel('Valor do Componente IDHM')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Componente', bbox_to_anchor=(1, 1))
ax.set_ylim(0.5, 0.85)

# Linha de referência: IDHM nacional médio
ax.axhline(df['IDHM'].mean(), color='gray', linestyle='--', linewidth=1.5,
           label='IDHM médio nacional')

plt.tight_layout()
plt.savefig('../modulo-4/figuras/fig8_componentes_idhm_regiao.png', bbox_inches='tight')
plt.show()

print('\n📖 NARRATIVA:')
print('   O componente Educação é consistentemente o menor em todas as regiões:')
print('   é o principal "gargalo" do IDHM brasileiro.')
print('   Já Longevidade é o componente mais alto — avanços na saúde pública')
print('   (SUS, vacinação) beneficiaram todo o Brasil de forma relativamente uniforme.')
print('   Renda é o componente com maior disparidade ENTRE regiões.')

---
## 🎯 8. Análise Multivariada — Visão Integrada

Até aqui analisamos variáveis uma a duas por vez. Agora vamos combinar múltiplas dimensões em um único gráfico para revelar padrões mais ricos.

In [ ]:
# Bubble chart: PIB per capita x IDHM x População x Região
# 4 variáveis em um único gráfico!

fig, ax = plt.subplots(figsize=(13, 8))

# Para não poluir o gráfico, amostramos 800 municípios aleatoriamente
df_amostra = df.dropna(subset=['GDP_CAPITA', 'IDHM', 'IBGE_POP', 'REGION']).copy()
df_amostra = df_amostra[df_amostra['GDP_CAPITA'] < df_amostra['GDP_CAPITA'].quantile(0.97)]
if len(df_amostra) > 800:
    df_amostra = df_amostra.sample(800, random_state=42)

# Tamanho da bolha = raiz quadrada da população (para não distorcer)
tamanho = (df_amostra['IBGE_POP'] / df_amostra['IBGE_POP'].max() * 1500 + 10)

for regiao in df_amostra['REGION'].unique():
    mask = df_amostra['REGION'] == regiao
    ax.scatter(
        df_amostra.loc[mask, 'GDP_CAPITA'],
        df_amostra.loc[mask, 'IDHM'],
        s=tamanho[mask], alpha=0.4,
        color=COR_REGIAO[regiao], label=regiao, edgecolors='none'
    )

ax.set_xlabel('PIB per Capita (R$)', fontsize=11)
ax.set_ylabel('IDHM', fontsize=11)
ax.set_title('Municípios Brasileiros: PIB per Capita × IDHM × População × Região\n'
             '(tamanho da bolha ∝ população)', fontsize=12, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x/1000:.0f}k'))
ax.legend(title='Região', framealpha=0.8)

# Quadrantes de análise
pib_median = df_amostra['GDP_CAPITA'].median()
idhm_median = df_amostra['IDHM'].median()
ax.axvline(pib_median, color='gray', linestyle=':', alpha=0.5)
ax.axhline(idhm_median, color='gray', linestyle=':', alpha=0.5)
ax.text(pib_median*1.05, df_amostra['IDHM'].max()*0.98, 
        'Ricos e\nDesenvolvidos', fontsize=8, color='gray')
ax.text(df_amostra['GDP_CAPITA'].min(), df_amostra['IDHM'].max()*0.98,
        'Pouco Ricos mas\nDesenvolvidos?', fontsize=8, color='gray')

plt.tight_layout()
plt.savefig('../modulo-4/figuras/fig9_bubble_chart.png', bbox_inches='tight')
plt.show()

---
## 📖 9. Storytelling com Dados — Construindo a Narrativa

> **Conceito-chave:** Storytelling com dados é a arte de transformar análises técnicas em narrativas claras e persuasivas para qualquer público.

### As 3 etapas do storytelling:

1. **Contexto**: Quem é o público? Qual é o problema?
2. **Conflito/Descoberta**: Qual foi o insight surpreendente?
3. **Resolução**: O que isso significa? O que pode ser feito?

---

### Painel Final: O Retrato do IDHM no Brasil

In [ ]:
fig = plt.figure(figsize=(16, 10))
fig.suptitle('O Retrato do Desenvolvimento Humano nos Municípios Brasileiros',
             fontsize=16, fontweight='bold', y=1.01)

# Layout: 2 linhas, 3 colunas
gs = fig.add_gridspec(2, 3, hspace=0.45, wspace=0.35)

# --- Painel 1: Histograma IDHM ---
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(df['IDHM'].dropna(), bins=30, color='steelblue', edgecolor='white', alpha=0.8)
ax1.axvline(df['IDHM'].mean(), color='red', linestyle='--', linewidth=1.5)
ax1.set_title('① Distribuição do IDHM', fontsize=10, fontweight='bold')
ax1.set_xlabel('IDHM'); ax1.set_ylabel('Municípios')

# --- Painel 2: Média por Região ---
ax2 = fig.add_subplot(gs[0, 1])
med_reg = df.groupby('REGION')['IDHM'].mean().sort_values()
cores_r = [COR_REGIAO[r] for r in med_reg.index]
ax2.barh(med_reg.index, med_reg.values, color=cores_r, edgecolor='white')
ax2.set_title('② IDHM Médio por Região', fontsize=10, fontweight='bold')
ax2.set_xlabel('IDHM Médio')
ax2.set_xlim(0.55, 0.80)
for i, v in enumerate(med_reg.values):
    ax2.text(v + 0.002, i, f'{v:.3f}', va='center', fontsize=8)

# --- Painel 3: Componentes nacionais ---
ax3 = fig.add_subplot(gs[0, 2])
if all(c in df.columns for c in ['IDHM_Renda','IDHM_Longevidade','IDHM_Educacao']):
    medias_comp = df[['IDHM_Renda','IDHM_Longevidade','IDHM_Educacao']].mean()
    medias_comp.index = ['Renda', 'Longevidade', 'Educação']
    cores_comp = ['#e6994c', '#56b4e9', '#009e73']
    barras = ax3.bar(medias_comp.index, medias_comp.values, color=cores_comp, edgecolor='white')
    for b, v in zip(barras, medias_comp.values):
        ax3.text(b.get_x()+b.get_width()/2, b.get_height()+0.003, f'{v:.3f}', ha='center', fontsize=9)
    ax3.set_ylim(0.5, 0.85)
ax3.set_title('③ Componentes do IDHM\n(Média Nacional)', fontsize=10, fontweight='bold')
ax3.set_ylabel('Valor')

# --- Painel 4: Scatter PIB x IDHM ---
ax4 = fig.add_subplot(gs[1, 0:2])
df_plot = df[df['GDP_CAPITA'] < df['GDP_CAPITA'].quantile(0.95)].sample(min(1000, len(df)), random_state=1)
for reg, grp in df_plot.groupby('REGION'):
    ax4.scatter(grp['GDP_CAPITA'], grp['IDHM'], color=COR_REGIAO[reg],
                alpha=0.3, s=12, label=reg)
ax4.set_title('④ PIB per Capita × IDHM por Região', fontsize=10, fontweight='bold')
ax4.set_xlabel('PIB per Capita (R$)')
ax4.set_ylabel('IDHM')
ax4.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'R${x/1000:.0f}k'))
ax4.legend(title='Região', fontsize=7, ncol=2)

# --- Painel 5: Desigualdade por Região (violinplot) ---
ax5 = fig.add_subplot(gs[1, 2])
ordem_reg = df.groupby('REGION')['IDHM'].median().sort_values().index.tolist()
dados_vio = [df[df['REGION']==r]['IDHM'].dropna().values for r in ordem_reg]
partes = ax5.violinplot(dados_vio, showmedians=True)
for pc, reg in zip(partes['bodies'], ordem_reg):
    pc.set_facecolor(COR_REGIAO[reg])
    pc.set_alpha(0.7)
ax5.set_xticks(range(1, len(ordem_reg)+1))
ax5.set_xticklabels([r[:3] for r in ordem_reg], fontsize=8)
ax5.set_title('⑤ Desigualdade Interna\npor Região', fontsize=10, fontweight='bold')
ax5.set_ylabel('IDHM')

plt.savefig('../modulo-4/figuras/fig10_painel_final.png', bbox_inches='tight')
plt.show()
print('✅ Painel-resumo gerado com sucesso!')

---
## ✅ 10. Conclusões da EDA

### O que aprendemos sobre o IDHM nos municípios brasileiros?

---

#### 🔑 Achado 1: Desigualdade Geográfica Persistente
Sul e Sudeste concentram os municípios com maior IDHM, enquanto Norte e Nordeste apresentam os menores índices. A diferença não é marginal — a distância entre a média da região Sul e do Nordeste supera 0,12 pontos no IDHM, o equivalente a anos de desenvolvimento.

#### 🔑 Achado 2: Educação é o Gargalo Nacional
Dos três componentes do IDHM, a **Educação** é consistentemente o mais baixo em todas as regiões. Isso indica que políticas educacionais têm o maior potencial de impacto para elevar o desenvolvimento humano no Brasil.

#### 🔑 Achado 3: Riqueza ≠ Desenvolvimento (necessariamente)
Embora exista correlação positiva entre PIB per capita e IDHM, há municípios com PIB elevado e IDHM moderado, e municípios pequenos com IDHM surpreendentemente alto. Isso sugere que a **distribuição** da riqueza importa tanto quanto seu volume.

#### 🔑 Achado 4: Desigualdade Dentro dos Estados
Vários estados apresentam altíssima variação interna no IDHM. Um estado pode ter uma capital com IDHM >0,80 e municípios interioranos com IDHM <0,55 — um abismo de desenvolvimento dentro do mesmo território.

#### 🔑 Achado 5: Brasil é um País de Pequenos Municípios
A grande maioria dos municípios tem menos de 20 mil habitantes. Essa estrutura fragmentada desafia políticas públicas de escala e exige soluções territorializadas.

---

### 💬 Reflexão Final

> *"Dados não falam por si só — precisamos fazer as perguntas certas. Esta EDA não encerra o debate sobre desenvolvimento humano no Brasil; ela o abre. Cada insight aqui é um convite a investigar mais a fundo: por que a educação ainda é um gargalo? Por que a renda não se traduz em IDHM em certos municípios? Quais políticas funcionaram nas exceções positivas?"*

---

### 📌 Próximos Passos Sugeridos
- Análise temporal: como o IDHM evoluiu entre 1991, 2000 e 2010?
- Modelagem preditiva: quais variáveis *causam* maior IDHM?
- Análise espacial: municípios vizinhos têm IDHM parecido? (autocorrelação espacial)
- Segmentação: clustering de municípios por perfil socioeconômico

---
## 📝 Recapitulando o que aprendemos

| Etapa | O que fizemos |
|-------|---------------|
| **0. Setup** | Importações, tema visual, paleta de cores |
| **1. Carregamento** | `.read_csv()`, `.shape`, `.info()`, `.describe()` |
| **2. Univariada** | Histograma, boxplot, faixas com `pd.cut()` |
| **3. Geográfica** | Agrupamento por região/estado, `.groupby().agg()` |
| **4. Bivariada** | Heatmap de correlação, scatter plot |
| **5. Desigualdade** | Desvio-padrão por grupo, barras agrupadas |
| **6. Porte** | `pd.cut()` para populações, barras com erro |
| **7. Componentes** | Barras agrupadas múltiplas variáveis |
| **8. Multivariada** | Bubble chart (4 variáveis simultâneas) |
| **9. Storytelling** | Painel de dashboard, narrativa integrada |
| **10. Conclusões** | Síntese em linguagem clara e acionável |

---
**🚀 Continue para o Projeto Final para aplicar tudo isso de forma autônoma!**